# Hawkes Calibration

This notebook is intentionally limited to orchestration and plots. The exponential and rough power-law likelihoods, simulators, calibration routines, and diagnostics live in `Hawkes.py`. See `workflow.md` before replacing the synthetic example with an empirical event proxy.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import expon

from Hawkes import (
    ExactHawkesCalibration,
    ExponentialHawkesCalibration,
    RoughHawkesCalibration,
)

plt.style.use("seaborn-v0_8-whitegrid")

## Event data

The default cell creates one reproducible rough-Hawkes sample so both kernels are compared on exactly the same events. Replace only `event_times` and `horizon` with the empirical event series.

In [ ]:
horizon = 250.0
true_tail_index = 0.45
true_cutoff = 0.25
true_branching_ratio = 0.55
true_alpha = (
    true_branching_ratio
    * true_tail_index
    * true_cutoff ** true_tail_index
)

event_times = RoughHawkesCalibration.simulate(
    lambda0=0.35,
    alpha=true_alpha,
    tail_index=true_tail_index,
    cutoff=true_cutoff,
    horizon=horizon,
    seed=20260614,
)

print(f"Events: {event_times.size}; observed rate: {event_times.size / horizon:.4f}")

## Fit both kernels

In [ ]:
exp_fit = ExponentialHawkesCalibration.fit(event_times, horizon)
rough_fit = RoughHawkesCalibration.fit(event_times, horizon)

comparison = pd.DataFrame(
    [
        {
            "model": exp_fit.model,
            "success": exp_fit.success,
            "log_likelihood": exp_fit.log_likelihood,
            "aic": exp_fit.aic,
            "bic": exp_fit.bic,
            **exp_fit.params,
        },
        {
            "model": rough_fit.model,
            "success": rough_fit.success,
            "log_likelihood": rough_fit.log_likelihood,
            "aic": rough_fit.aic,
            "bic": rough_fit.bic,
            **rough_fit.params,
        },
    ]
)
comparison

## Fitted conditional intensities

In [ ]:
grid = np.linspace(0.0, horizon, 2500)
ep = exp_fit.params
rp = rough_fit.params

exp_intensity = ExponentialHawkesCalibration.intensity_on_grid(
    grid, event_times, ep["lambda0"], ep["alpha"], ep["beta"]
)
rough_intensity = RoughHawkesCalibration.intensity_on_grid(
    grid, event_times, rp["lambda0"], rp["alpha"],
    rp["tail_index"], rp["cutoff"]
)

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(grid, exp_intensity, label="Exponential fit", linewidth=1.7)
ax.plot(grid, rough_intensity, label="Rough power-law fit", linewidth=1.7)
ax.vlines(event_times, 0.0, 0.04 * max(rough_intensity.max(), exp_intensity.max()),
          color="black", alpha=0.25, linewidth=0.6, label="Events")
ax.set(xlabel="Time", ylabel="Conditional intensity", title="Fitted Hawkes intensities")
ax.legend()
plt.show()

## Kernel decay

In [ ]:
lags = np.geomspace(1e-3, max(20.0, horizon / 5.0), 500)
exp_kernel = ExponentialHawkesCalibration.kernel(lags, ep["alpha"], ep["beta"])
rough_kernel = RoughHawkesCalibration.kernel(
    lags, rp["alpha"], rp["tail_index"], rp["cutoff"]
)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.loglog(lags, exp_kernel, label="Exponential kernel", linewidth=2.0)
ax.loglog(lags, rough_kernel, label="Rough power-law kernel", linewidth=2.0)
ax.set(xlabel="Lag", ylabel="Kernel value", title="Estimated excitation decay")
ax.legend()
plt.show()

## Time-rescaling residuals

Under a correctly specified point process, compensator increments should be approximately `Exp(1)`.

In [ ]:
exp_residuals = ExponentialHawkesCalibration.time_rescaling_residuals(
    event_times, ep["lambda0"], ep["alpha"], ep["beta"]
)
rough_residuals = RoughHawkesCalibration.time_rescaling_residuals(
    event_times, rp["lambda0"], rp["alpha"], rp["tail_index"], rp["cutoff"]
)

def empirical_cdf(values):
    ordered = np.sort(values)
    probability = np.arange(1, ordered.size + 1) / ordered.size
    return ordered, probability

x_exp, y_exp = empirical_cdf(exp_residuals)
x_rough, y_rough = empirical_cdf(rough_residuals)
x_theory = np.linspace(0.0, max(x_exp.max(), x_rough.max()), 500)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.step(x_exp, y_exp, where="post", label="Exponential residuals")
ax.step(x_rough, y_rough, where="post", label="Rough residuals")
ax.plot(x_theory, expon.cdf(x_theory), "k--", label="Exp(1) CDF")
ax.set(xlabel="Compensator increment", ylabel="CDF", title="Time-rescaling diagnostic")
ax.legend()
plt.show()

## Exact affine option calibration

The expensive option-surface calibration is disabled by default. Its optimizer is implemented by `ExactHawkesCalibration` in `Hawkes.py`; `BatesHawkesExact.py` is only the pricing engine.

In [ ]:
RUN_EXACT_OPTION_CALIBRATION = False

if RUN_EXACT_OPTION_CALIBRATION:
    import json
    from pathlib import Path
    from BnS import BnS

    data_dir = Path("Data")
    metadata = json.loads((data_dir / "gld_chain_wide_meta.json").read_text())
    option_data = pd.read_csv(data_dir / "gld_iv_dataset_chebyshev.csv")
    spot = float(metadata["S0"])
    dividend_yield = 0.0
    option_data["vega"] = [
        BnS.calculate_bs_vega(
            spot, row.K, row.T, row.rate, dividend_yield, row.implied_vol
        )
        for row in option_data.itertuples(index=False)
    ]
    exact_result = ExactHawkesCalibration.calibrate_constvol(
        option_data, spot, q=dividend_yield, maxiter=35, popsize=8, seed=20260614
    )
    exact_result.x